<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_3_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.3-B — COVERAGE vs LABEL PRECISION
# RESUMABLE / PERSISTENT GOOGLE-DRIVE VERSION
#
# Fixed MC budget:
#   (R,m) = (200,250), (500,100), (1000,50), (2000,25), (5000,10)
#   R*m = 50,000
#
# PERSISTENCE:
#   1. Google Drive permanent storage
#   2. exact-target chunks saved permanently
#   3. Monte-Carlo chunks saved permanently
#   4. neural training checkpoint every 5 epochs
#   5. optimizer + scheduler + RNG state saved
#   6. result saved after every completed allocation
#
# AFTER COLAB DISCONNECT:
#   -> rerun THIS SAME CELL
#   -> completed work is loaded from Drive
#   -> interrupted training resumes from its checkpoint
#
# OUTPUTS NEEDED FOR 5.3-B:
#   1 table
#   1 two-panel figure
#
# CPU ONLY
# Sparse LU only — NO matrix inverse
# =====================================================================================


# =====================================================================================
# 0. MOUNT GOOGLE DRIVE + CPU ENVIRONMENT
# =====================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os

os.environ["OMP_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
os.environ["NUMEXPR_NUM_THREADS"]="1"

import time
import math
import random
import pickle
import shutil

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

from joblib import Parallel, delayed

from numba import njit, prange, set_num_threads, get_num_threads

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# =====================================================================================
# 1. CONFIG
# =====================================================================================

@dataclass
class C:

    seed:int=20260820

    beta:Tuple[float,float]=(.30,1.50)
    gamma:Tuple[float,float]=(.20,1.00)
    omega:Tuple[float,float]=(.02,.50)
    frac:Tuple[float,float]=(.02,.20)

    trainN:Tuple[int,...]=tuple(
        range(40,401,20)
    )

    width:int=128
    depth:int=3

    batch:int=64
    epochs:int=300

    lr:float=1e-3
    wd:float=1e-6

    patience:int=20
    delta:float=1e-6
    clip:float=5.


cfg=C()


ALLOC=(
    (200,250),
    (500,100),
    (1000,50),
    (2000,25),
    (5000,10)
)

BUDGET=50_000

assert all(
    R*m==BUDGET
    for R,m in ALLOC
)

RMAX=max(
    R
    for R,m in ALLOC
)

Nscale=max(
    cfg.trainN
)

TEST_N=tuple(
    range(40,401,10)
)


# =====================================================================================
# 2. PERMANENT GOOGLE-DRIVE DIRECTORIES
# =====================================================================================

# Change only this path if you want another Drive folder.
ROOT=Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_3B_budget50000_v3"
)

CACHE=ROOT/"cache"
EXACT_CACHE=CACHE/"exact"
MC_CACHE=CACHE/"mc"
MODEL_CACHE=CACHE/"models"
CHECKPOINT_CACHE=CACHE/"checkpoints"

OUT=ROOT/"results"

for d in (
    ROOT,
    CACHE,
    EXACT_CACHE,
    MC_CACHE,
    MODEL_CACHE,
    CHECKPOINT_CACHE,
    OUT
):
    d.mkdir(
        parents=True,
        exist_ok=True
    )


print("="*90)
print("PERSISTENT DIRECTORY")
print(ROOT)
print("="*90)


# =====================================================================================
# 3. OPTIONAL MIGRATION OF USEFUL FILES FROM CURRENT /content SESSION
#
# If the old Colab session still contains completed exact caches/model,
# copy them permanently to Google Drive once.
#
# MC caches/models are deliberately NOT migrated because the MC kernel
# has now been changed to the overflow-stopping implementation.
# =====================================================================================

OLD_CACHE=Path(
    "/content/cache_5_3B_CPU_budget50000"
)

if OLD_CACHE.exists():

    legacy_map={

        "exact_train_R5000.pkl":
            EXACT_CACHE/"TRAIN_full.pkl",

        "exact_val_400.pkl":
            EXACT_CACHE/"VAL_full.pkl",

        "exact_test_700.pkl":
            EXACT_CACHE/"TEST_full.pkl",

        "model_exact_R5000_CPU.pt":
            MODEL_CACHE/"model_exact_R5000_CPU.pt"
    }


    for oldname,newfile in legacy_map.items():

        oldfile=OLD_CACHE/oldname

        if (
            oldfile.exists()
            and
            not newfile.exists()
        ):

            shutil.copy2(
                oldfile,
                newfile
            )

            print(
                "Migrated to Drive:",
                oldname
            )


# =====================================================================================
# 4. ATOMIC FILE WRITES
#
# Avoid leaving a half-written checkpoint if Colab disconnects during saving.
# =====================================================================================

def atomic_pickle(
    obj,
    path
):

    path=Path(path)

    tmp=path.with_suffix(
        path.suffix+".tmp"
    )

    with open(
        tmp,
        "wb"
    ) as f:

        pickle.dump(
            obj,
            f,
            pickle.HIGHEST_PROTOCOL
        )

    os.replace(
        tmp,
        path
    )


def atomic_torch_save(
    obj,
    path
):

    path=Path(path)

    tmp=path.with_suffix(
        path.suffix+".tmp"
    )

    torch.save(
        obj,
        tmp
    )

    os.replace(
        tmp,
        path
    )


def safe_pickle_load(
    path,
    default=None
):

    try:

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(f)

    except Exception:

        return default


# =====================================================================================
# 5. CPU SETTINGS
# =====================================================================================

CPU=os.cpu_count() or 1

# Sparse LU is RAM intensive.
N_EXACT=min(
    2,
    CPU
)

# Numba
N_MC=min(
    CPU,
    get_num_threads()
)

set_num_threads(
    N_MC
)

# PyTorch
TORCH_THREADS=min(
    8,
    CPU
)

torch.set_num_threads(
    TORCH_THREADS
)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass


def seed_all(s):

    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)


seed_all(
    cfg.seed
)


print(
    "CPU cores:",
    CPU
)

print(
    "Exact workers:",
    N_EXACT
)

print(
    "Numba threads:",
    N_MC
)

print(
    "PyTorch threads:",
    torch.get_num_threads()
)


# =====================================================================================
# 6. DATA RECORD
# =====================================================================================

@dataclass
class Rec:

    b:float
    g:float
    w:float

    N:int
    i0:int

    p:np.ndarray


# =====================================================================================
# 7. CACHED SIRS TOPOLOGY
# =====================================================================================

@lru_cache(None)
def topo(N):

    st=[
        (s,i)
        for i in range(1,N+1)
        for s in range(N-i+1)
    ]

    ix={
        x:j
        for j,x in enumerate(st)
    }

    M=len(st)

    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]

    db=np.zeros(M)
    dg=np.zeros(M)
    dw=np.zeros(M)
    qb=np.zeros(M)


    for j,(s,i) in enumerate(st):

        r=N-s-i

        # infection
        if s:

            ir.append(j)
            ic.append(
                ix[(s-1,i+1)]
            )

            rate=s*i/N

            ib.append(rate)
            db[j]=rate


        # recovery
        dg[j]=i

        if i==1:

            qb[j]=i

        else:

            rr.append(j)
            rc.append(
                ix[(s,i-1)]
            )
            rb.append(i)


        # immunity loss
        if r:

            wr.append(j)
            wc.append(
                ix[(s+1,i)]
            )
            wb.append(r)
            dw[j]=r


    A=lambda x,d=float:np.asarray(
        x,
        dtype=d
    )


    return (

        ix,M,

        A(ir,int),
        A(ic,int),
        A(ib),

        A(rr,int),
        A(rc,int),
        A(rb),

        A(wr,int),
        A(wc,int),
        A(wb),

        db,dg,dw,qb
    )


# =====================================================================================
# 8. EXACT DISTRIBUTION
# =====================================================================================

def exact_p(
    b,g,w,N,i0
):

    (
        ix,M,
        ir,ic,ib,
        rr,rc,rb,
        wr,wc,wb,
        db,dg,dw,qb
    )=topo(N)


    rows=np.r_[
        ir,rr,wr,np.arange(M)
    ]

    cols=np.r_[
        ic,rc,wc,np.arange(M)
    ]


    T=sparse.coo_matrix(

        (

            np.r_[

                b*ib,
                g*rb,
                w*wb,

                -(
                    b*db
                    +
                    g*dg
                    +
                    w*dw
                )
            ],

            (
                rows,
                cols
            )
        ),

        shape=(M,M)

    ).tocsc()


    D1=sparse.coo_matrix(

        (
            b*ib,
            (
                ir,
                ic
            )
        ),

        shape=(M,M)

    ).tocsc()


    A0=(
        -(T-D1)
    ).tocsc()


    lu=splu(
        A0,
        permc_spec="COLAMD"
    )


    q=g*qb


    v=np.zeros(M)

    v[
        ix[(N-i0,i0)]
    ]=1.


    bvec=lu.solve(q)

    D1T=D1.T.tocsr()


    p=np.zeros(
        N+2
    )


    for k in range(
        N+1
    ):

        p[k]=v@bvec

        v=np.asarray(

            D1T
            @
            lu.solve(
                v,
                trans="T"
            )

        ).ravel()


    p[-1]=v.sum()

    p[
        np.abs(p)<1e-12
    ]=0.

    p=np.maximum(
        p,
        0.
    )


    mass=p.sum()


    if (
        not np.isfinite(mass)
        or
        mass<=0
    ):

        raise RuntimeError(
            "Invalid exact PMF"
        )


    return p/mass


# =====================================================================================
# 9. DESIGN
# =====================================================================================

def design(
    n,
    Ns,
    seed
):

    U=qmc.LatinHypercube(
        4,
        seed=seed
    ).random(n)


    scale=lambda x,a:(
        a[0]
        +
        (a[1]-a[0])*x
    )


    b=scale(
        U[:,0],
        cfg.beta
    )

    g=scale(
        U[:,1],
        cfg.gamma
    )

    w=scale(
        U[:,2],
        cfg.omega
    )

    f=scale(
        U[:,3],
        cfg.frac
    )


    Nv=np.tile(

        np.asarray(Ns),

        math.ceil(
            n/len(Ns)
        )

    )[:n]


    rng=np.random.default_rng(
        seed+99
    )

    rng.shuffle(
        Nv
    )


    i0=np.asarray([

        int(
            np.clip(
                round(
                    f[j]*Nv[j]
                ),
                2,
                Nv[j]
            )
        )

        for j in range(n)

    ])


    for N in Ns:

        z=np.where(
            Nv==N
        )[0]

        if len(z):

            k=max(
                1,
                round(.25*len(z))
            )

            i0[
                rng.choice(
                    z,
                    k,
                    replace=False
                )
            ]=1


    return [

        (
            float(b[j]),
            float(g[j]),
            float(w[j]),
            int(Nv[j]),
            int(i0[j])
        )

        for j in range(n)

    ]


# =====================================================================================
# 10. RESUMABLE EXACT TARGET GENERATION
#
# Each chunk is stored permanently on Google Drive.
# Disconnect after chunk 37? Next run starts with chunk 38.
# =====================================================================================

EXACT_CHUNK=100


def _exact_one(
    j,
    x
):

    return (
        j,
        Rec(
            *x,
            exact_p(*x)
        )
    )


def exact_set_resumable(
    configs,
    name
):

    full_file=(
        EXACT_CACHE
        /
        f"{name}_full.pkl"
    )


    # -------------------------------------------------------------------------
    # Entire dataset already finished
    # -------------------------------------------------------------------------

    if full_file.exists():

        ans=safe_pickle_load(
            full_file
        )

        if (
            ans is not None
            and
            len(ans)==len(configs)
        ):

            print(
                f"{name}: full Drive cache loaded "
                f"({len(ans):,})"
            )

            return ans


    folder=(
        EXACT_CACHE
        /
        name
    )

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    pieces=[]


    for start in range(
        0,
        len(configs),
        EXACT_CHUNK
    ):

        end=min(
            start+EXACT_CHUNK,
            len(configs)
        )


        file=(
            folder
            /
            f"chunk_{start:05d}_{end:05d}.pkl"
        )


        part=None


        if file.exists():

            part=safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                len(part)==end-start
            ):

                print(
                    f"{name} {start:5d}:{end:5d} | "
                    "Drive cache"
                )

            else:

                part=None


        if part is None:

            print(
                f"{name} {start:5d}:{end:5d} | "
                f"computing with {N_EXACT} workers"
            )


            jobs=list(
                enumerate(
                    configs[start:end]
                )
            )


            jobs.sort(
                key=lambda z:
                z[1][3],
                reverse=True
            )


            t0=time.perf_counter()


            result=Parallel(

                n_jobs=N_EXACT,

                backend="threading"

            )(

                delayed(
                    _exact_one
                )(
                    j,
                    x
                )

                for j,x in jobs

            )


            result.sort(
                key=lambda z:z[0]
            )


            part=[
                r
                for _,r in result
            ]


            atomic_pickle(
                part,
                file
            )


            print(
                f"  saved | "
                f"{time.perf_counter()-t0:.1f}s"
            )


        pieces.extend(
            part
        )


    atomic_pickle(
        pieces,
        full_file
    )


    print(
        f"{name}: COMPLETE and permanently cached."
    )


    return pieces


# =====================================================================================
# 11. MONTE CARLO — EXACT EARLY STOP WHEN OVERFLOW IS KNOWN
#
# We need only:
#   p(0),...,p(N),p(>N)
#
# Once C=N+1, the final category is already known.
# =====================================================================================

@njit
def sim_C(
    b,g,w,N,i0
):

    S=N-i0
    I=i0
    R=0

    C=0


    while I>0:

        inf=b*S*I/N
        rec=g*I
        wan=w*R

        z=np.random.random()*(
            inf+rec+wan
        )


        if z<inf:

            S-=1
            I+=1
            C+=1

            # ---------------------------------------------------------
            # Exact early termination for overflow category
            # ---------------------------------------------------------

            if C>=N+1:

                return N+1


        elif z<inf+rec:

            I-=1
            R+=1


        else:

            R-=1
            S+=1


    return C


# =====================================================================================
# 12. RESUMABLE NESTED MONTE CARLO
# =====================================================================================

MC_LEVELS=np.asarray(
    [10,25,50,100,250],
    dtype=np.int64
)

MC_CHUNK=250


@njit(parallel=True)
def nested_mc_chunk(
    B,G,W,N,I0,
    maxm,
    maxN,
    global_start,
    seed
):

    n=len(N)


    levels=np.asarray(
        [10,25,50,100,250],
        dtype=np.int64
    )


    snap=np.zeros(
        (
            5,
            n,
            maxN+2
        ),
        dtype=np.int32
    )


    for j in prange(n):

        global_j=(
            global_start+j
        )


        np.random.seed(
            seed
            +
            100003*global_j
        )


        h=np.zeros(
            maxN+2,
            dtype=np.int32
        )


        lev=0


        for q in range(
            1,
            maxm[j]+1
        ):

            c=sim_C(
                B[j],
                G[j],
                W[j],
                N[j],
                I0[j]
            )


            h[c]+=1


            if (
                lev<5
                and
                q==levels[lev]
            ):

                snap[
                    lev,
                    j,
                    :
                ]=h


                lev+=1


    return snap


def make_mc_labels_resumable(
    records
):

    final_file=(
        MC_CACHE
        /
        "nested_MC_budget50000_overflow_v3.pkl"
    )


    # -------------------------------------------------------------------------
    # Finished labels already available
    # -------------------------------------------------------------------------

    if final_file.exists():

        ans=safe_pickle_load(
            final_file
        )

        if ans is not None:

            print(
                "Final nested MC labels loaded from Drive."
            )

            return ans


    B=np.asarray(
        [r.b for r in records],
        dtype=np.float64
    )

    G=np.asarray(
        [r.g for r in records],
        dtype=np.float64
    )

    W=np.asarray(
        [r.w for r in records],
        dtype=np.float64
    )

    N=np.asarray(
        [r.N for r in records],
        dtype=np.int64
    )

    I0=np.asarray(
        [r.i0 for r in records],
        dtype=np.int64
    )


    maxm=np.zeros(
        RMAX,
        dtype=np.int64
    )


    maxm[:200]=250
    maxm[200:500]=100
    maxm[500:1000]=50
    maxm[1000:2000]=25
    maxm[2000:5000]=10


    print("\n"+"="*85)
    print("RESUMABLE NESTED MONTE CARLO")
    print("="*85)

    print(
        "Nominal nested trajectories:",
        f"{maxm.sum():,}"
    )

    print(
        "Early overflow termination:",
        "ON"
    )


    # -------------------------------------------------------------------------
    # Compile Numba once
    # -------------------------------------------------------------------------

    _=nested_mc_chunk(

        np.asarray([.8]),
        np.asarray([.5]),
        np.asarray([.1]),

        np.asarray(
            [40],
            dtype=np.int64
        ),

        np.asarray(
            [1],
            dtype=np.int64
        ),

        np.asarray(
            [10],
            dtype=np.int64
        ),

        40,
        0,
        cfg.seed+999
    )


    pieces=[]


    for start in range(
        0,
        RMAX,
        MC_CHUNK
    ):

        end=min(
            start+MC_CHUNK,
            RMAX
        )


        file=(
            MC_CACHE
            /
            f"chunk_{start:05d}_{end:05d}.pkl"
        )


        part=None


        if file.exists():

            part=safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                part.shape[1]==end-start
            ):

                print(
                    f"MC {start:5d}:{end:5d} | "
                    "Drive cache"
                )

            else:

                part=None


        if part is None:

            t0=time.perf_counter()


            part=nested_mc_chunk(

                B[start:end],
                G[start:end],
                W[start:end],
                N[start:end],
                I0[start:end],

                maxm[start:end],

                max(cfg.trainN),

                start,

                cfg.seed+999

            )


            atomic_pickle(
                part,
                file
            )


            print(
                f"MC {start:5d}:{end:5d} | "
                f"{time.perf_counter()-t0:.1f}s | "
                "saved permanently"
            )


        pieces.append(
            part
        )


    snap=np.concatenate(
        pieces,
        axis=1
    )


    level={
        10:0,
        25:1,
        50:2,
        100:3,
        250:4
    }


    labels={}


    for R,m in ALLOC:

        q=level[m]


        labels[(R,m)]=[

            snap[
                q,
                j,
                :records[j].N+2
            ].astype(
                np.float32
            )
            /
            m

            for j in range(R)

        ]


    atomic_pickle(
        labels,
        final_file
    )


    print(
        "Nested MC complete and permanently saved."
    )


    return labels


# =====================================================================================
# 13. HAZARD NETWORK
# =====================================================================================

class HazardNet(
    nn.Module
):

    def __init__(self):

        super().__init__()


        L=[]

        d=6


        for _ in range(
            cfg.depth
        ):

            L += [

                nn.Linear(
                    d,
                    cfg.width
                ),

                nn.SiLU()

            ]

            d=cfg.width


        L.append(
            nn.Linear(
                d,
                1
            )
        )


        self.net=nn.Sequential(
            *L
        )


    def forward(
        self,
        x
    ):

        return torch.sigmoid(
            self.net(
                x
            ).squeeze(-1)
        )


# =====================================================================================
# 14. SAME-N CPU PACKING
# =====================================================================================

def pack(
    records,
    labels=None
):

    groups={}


    for j,r in enumerate(records):

        groups.setdefault(
            r.N,
            []
        ).append(j)


    P={}


    for N,idx in groups.items():

        idx=np.asarray(
            idx
        )


        B=len(idx)

        K=N+1


        b=torch.tensor(
            [records[j].b for j in idx],
            dtype=torch.float32
        )[:,None]


        g=torch.tensor(
            [records[j].g for j in idx],
            dtype=torch.float32
        )[:,None]


        w=torch.tensor(
            [records[j].w for j in idx],
            dtype=torch.float32
        )[:,None]


        ns=torch.full(
            (B,1),
            N/Nscale,
            dtype=torch.float32
        )


        i0=torch.tensor(
            [
                records[j].i0/N
                for j in idx
            ],
            dtype=torch.float32
        )[:,None]


        c=(
            torch.arange(
                K,
                dtype=torch.float32
            )
            /
            N
        )[None,:]


        X=torch.stack(

            [

                b.expand(B,K),
                g.expand(B,K),
                w.expand(B,K),
                ns.expand(B,K),
                i0.expand(B,K),
                c.expand(B,K)

            ],

            dim=2

        ).contiguous()


        Y=np.stack([

            records[j].p
            if labels is None
            else labels[j]

            for j in idx

        ]).astype(
            np.float32
        )


        P[N]={

            "X":
                X,

            "Y":
                torch.from_numpy(Y),

            "n":
                B

        }


    return P


# =====================================================================================
# 15. PMF + TAIL + LOSS
# =====================================================================================

def pmf_from_h(
    h
):

    B=h.shape[0]


    surv=torch.cat(

        [

            torch.ones(
                (B,1),
                dtype=h.dtype
            ),

            torch.cumprod(
                1-h[:,:-1],
                dim=1
            )

        ],

        dim=1

    )


    return torch.cat(

        [

            surv*h,

            torch.prod(
                1-h,
                dim=1,
                keepdim=True
            )

        ],

        dim=1

    )


def tail(
    P
):

    return torch.flip(

        torch.cumsum(

            torch.flip(
                P[:,1:],
                dims=[1]
            ),

            dim=1

        ),

        dims=[1]

    )


def predict_batch(
    net,
    X
):

    B,K,_=X.shape


    h=net(

        X.reshape(
            B*K,
            6
        )

    ).reshape(
        B,
        K
    )


    return pmf_from_h(
        h
    )


def loss_batch(
    net,
    X,
    Y
):

    P=predict_batch(
        net,
        X
    )


    Lp=torch.sum(
        (P-Y)**2,
        dim=1
    )


    Lrho=torch.mean(
        (
            tail(P)
            -
            tail(Y)
        )**2,
        dim=1
    )


    return (
        Lp
        +
        Lrho
    ).mean()


# =====================================================================================
# 16. MINI-BATCHES / VALIDATION
# =====================================================================================

def batches(
    P,
    rng,
    shuffle=True
):

    jobs=[]


    for N,G in P.items():

        idx=np.arange(
            G["n"]
        )


        if shuffle:

            rng.shuffle(
                idx
            )


        for s in range(
            0,
            len(idx),
            cfg.batch
        ):

            jobs.append(
                (
                    N,
                    idx[
                        s:
                        s+cfg.batch
                    ]
                )
            )


    if shuffle:

        rng.shuffle(
            jobs
        )


    return jobs


@torch.no_grad()
def val_loss(
    net,
    P
):

    net.eval()


    total=0.

    n=0


    rng=np.random.default_rng(
        1
    )


    for N,idx in batches(
        P,
        rng,
        False
    ):

        L=loss_batch(

            net,

            P[N]["X"][idx],

            P[N]["Y"][idx]

        )


        total+=(
            L.item()
            *
            len(idx)
        )


        n+=len(idx)


    return total/n


# =====================================================================================
# 17. RESUMABLE NEURAL TRAINING
#
# Permanent checkpoint every CHECKPOINT_EVERY epochs.
#
# Saves:
#   - current model
#   - optimizer
#   - scheduler
#   - best model
#   - best epoch
#   - early-stopping counter
#   - mini-batch RNG state
#   - torch RNG state
#   - elapsed training time
# =====================================================================================

CHECKPOINT_EVERY=5


def fit_resumable(
    TR,
    VA,
    seed,
    name
):

    final_file=(
        MODEL_CACHE
        /
        f"{name}_FINAL.pt"
    )


    ckpt_file=(
        CHECKPOINT_CACHE
        /
        f"{name}_CHECKPOINT.pt"
    )


    # -------------------------------------------------------------------------
    # Completed model already exists
    # -------------------------------------------------------------------------

    if final_file.exists():

        x=torch.load(
            final_file,
            map_location="cpu",
            weights_only=False
        )


        net=HazardNet()

        net.load_state_dict(
            x["state"]
        )


        print(
            f"{name}: FINAL model loaded | "
            f"epoch={x['best_epoch']}"
        )


        return (
            net,
            x.get(
                "training_sec",
                0.
            )
        )


    # -------------------------------------------------------------------------
    # Initial setup
    # -------------------------------------------------------------------------

    seed_all(
        seed
    )


    net=HazardNet()


    optimizer=torch.optim.AdamW(

        net.parameters(),

        lr=cfg.lr,

        weight_decay=cfg.wd

    )


    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        factor=.5,

        patience=15

    )


    rng=np.random.default_rng(
        seed+77
    )


    start_epoch=1

    best=np.inf

    best_epoch=0

    best_state=None

    wait=0

    previous_elapsed=0.


    # -------------------------------------------------------------------------
    # Resume interrupted training
    # -------------------------------------------------------------------------

    if ckpt_file.exists():

        try:

            ck=torch.load(

                ckpt_file,

                map_location="cpu",

                weights_only=False

            )


            net.load_state_dict(
                ck["model"]
            )


            optimizer.load_state_dict(
                ck["optimizer"]
            )


            scheduler.load_state_dict(
                ck["scheduler"]
            )


            start_epoch=(
                ck["epoch"]
                +
                1
            )


            best=ck["best"]

            best_epoch=ck["best_epoch"]

            best_state=ck["best_state"]

            wait=ck["wait"]

            previous_elapsed=ck.get(
                "elapsed_sec",
                0.
            )


            rng.bit_generator.state=(
                ck["rng_state"]
            )


            if "torch_rng" in ck:

                torch.set_rng_state(
                    ck["torch_rng"]
                )


            print(
                f"{name}: RESUMING from epoch "
                f"{start_epoch}"
            )


            print(
                f"  best epoch={best_epoch} | "
                f"best val={best:.4e} | "
                f"wait={wait}"
            )


        except Exception as e:

            print(
                "Checkpoint could not be loaded; "
                "starting this model again."
            )

            print(
                e
            )


    session_start=time.perf_counter()


    # -------------------------------------------------------------------------
    # Training
    # -------------------------------------------------------------------------

    for epoch in range(
        start_epoch,
        cfg.epochs+1
    ):

        net.train()


        for N,idx in batches(
            TR,
            rng,
            True
        ):

            X=TR[N]["X"][idx]

            Y=TR[N]["Y"][idx]


            optimizer.zero_grad(
                set_to_none=True
            )


            L=loss_batch(
                net,
                X,
                Y
            )


            L.backward()


            torch.nn.utils.clip_grad_norm_(
                net.parameters(),
                cfg.clip
            )


            optimizer.step()


        v=val_loss(
            net,
            VA
        )


        scheduler.step(
            v
        )


        if (
            best_state is None
            or
            v<best-cfg.delta
        ):

            best=v

            best_epoch=epoch

            wait=0


            best_state={

                k:
                x.detach().clone()

                for k,x
                in net.state_dict().items()

            }


        else:

            wait+=1


        # ---------------------------------------------------------------------
        # Persistent periodic checkpoint
        # ---------------------------------------------------------------------

        if (
            epoch%CHECKPOINT_EVERY==0
            or
            epoch==1
        ):

            elapsed_total=(

                previous_elapsed

                +

                (
                    time.perf_counter()
                    -
                    session_start
                )

            )


            atomic_torch_save(

                {

                    "epoch":
                        epoch,

                    "model":
                        net.state_dict(),

                    "optimizer":
                        optimizer.state_dict(),

                    "scheduler":
                        scheduler.state_dict(),

                    "best":
                        best,

                    "best_epoch":
                        best_epoch,

                    "best_state":
                        best_state,

                    "wait":
                        wait,

                    "rng_state":
                        rng.bit_generator.state,

                    "torch_rng":
                        torch.get_rng_state(),

                    "elapsed_sec":
                        elapsed_total

                },

                ckpt_file

            )


            print(
                f"{name} | "
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait} | "
                "checkpoint saved"
            )


        elif epoch%20==0:

            print(
                f"{name} | "
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait}"
            )


        # ---------------------------------------------------------------------
        # Early stopping
        # ---------------------------------------------------------------------

        if wait>=cfg.patience:

            print(
                f"{name}: early stopping at epoch {epoch}"
            )

            break


    # -------------------------------------------------------------------------
    # Finalize model
    # -------------------------------------------------------------------------

    elapsed_total=(

        previous_elapsed

        +

        (
            time.perf_counter()
            -
            session_start
        )

    )


    net.load_state_dict(
        best_state
    )


    atomic_torch_save(

        {

            "state":
                best_state,

            "best_epoch":
                best_epoch,

            "validation":
                best,

            "training_sec":
                elapsed_total

        },

        final_file

    )


    # Finished checkpoint no longer needed
    if ckpt_file.exists():

        ckpt_file.unlink()


    print(
        f"{name}: COMPLETE | "
        f"best epoch={best_epoch} | "
        f"best val={best:.4e}"
    )


    return (
        net,
        elapsed_total
    )


# =====================================================================================
# 18. TEST METRICS
# =====================================================================================

@torch.no_grad()
def metrics(
    net,
    P
):

    net.eval()


    E2=[]

    Erho=[]


    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            X=G["X"][
                s:
                s+cfg.batch
            ]


            Y=G["Y"][
                s:
                s+cfg.batch
            ]


            Ph=predict_batch(
                net,
                X
            )


            E2.append(

                torch.linalg.vector_norm(
                    Ph-Y,
                    dim=1
                ).numpy()

            )


            Erho.append(

                torch.max(

                    torch.abs(
                        tail(Ph)
                        -
                        tail(Y)
                    ),

                    dim=1

                ).values.numpy()

            )


    return (
        np.concatenate(E2),
        np.concatenate(Erho)
    )


# =====================================================================================
# 19. GENERATE / LOAD EXACT DATA
# =====================================================================================

print("\nGenerating deterministic designs...")


train_design=design(
    RMAX,
    cfg.trainN,
    cfg.seed+1
)


val_design=design(
    400,
    cfg.trainN,
    cfg.seed+2
)


test_design=design(
    700,
    TEST_N,
    cfg.seed+3
)


print("\nExact targets...")


train=exact_set_resumable(
    train_design,
    "TRAIN"
)


val=exact_set_resumable(
    val_design,
    "VAL"
)


test=exact_set_resumable(
    test_design,
    "TEST"
)


VA=pack(
    val
)


TE=pack(
    test
)


# =====================================================================================
# 20. PERSISTENT EXPERIMENT PROGRESS
# =====================================================================================

PROGRESS_FILE=(
    ROOT
    /
    "experiment_progress.pkl"
)


progress=safe_pickle_load(
    PROGRESS_FILE,
    default=None
)


if progress is None:

    progress={

        "exact":
            None,

        "mc":
            {}

    }


# =====================================================================================
# 21. EXACT-TEACHER REFERENCE
# =====================================================================================

if progress["exact"] is None:

    print("\n"+"="*80)
    print("EXACT TEACHER | R=5000")
    print("="*80)


    TR=pack(
        train
    )


    exact_net,exact_training_sec=fit_resumable(

        TR,

        VA,

        cfg.seed+7001,

        "exact_R5000"

    )


    ee,er=metrics(
        exact_net,
        TE
    )


    progress["exact"]={

        "E2_med":
            float(
                np.median(ee)
            ),

        "E2_q1":
            float(
                np.quantile(ee,.25)
            ),

        "E2_q3":
            float(
                np.quantile(ee,.75)
            ),

        "E2_p95":
            float(
                np.quantile(ee,.95)
            ),

        "Erho_med":
            float(
                np.median(er)
            ),

        "Erho_q1":
            float(
                np.quantile(er,.25)
            ),

        "Erho_q3":
            float(
                np.quantile(er,.75)
            ),

        "Erho_p95":
            float(
                np.quantile(er,.95)
            )

    }


    atomic_pickle(
        progress,
        PROGRESS_FILE
    )


    print(
        "Exact-reference metrics permanently saved."
    )


    del TR,exact_net


else:

    print(
        "\nExact-reference experiment already complete."
    )


EX=progress["exact"]


# =====================================================================================
# 22. GENERATE / LOAD RESUMABLE MC LABELS
# =====================================================================================

labels_all=make_mc_labels_resumable(
    train
)


# =====================================================================================
# 23. FIXED-BUDGET MC ALLOCATIONS — RESUMABLE
# =====================================================================================

for R,m in ALLOC:

    key=f"R{R}_m{m}"


    # -------------------------------------------------------------------------
    # Allocation already completed
    # -------------------------------------------------------------------------

    if key in progress["mc"]:

        print(
            f"\n{key}: result already completed — skipping."
        )

        continue


    print("\n"+"="*80)

    print(
        f"MC TEACHER | "
        f"R={R:,} | "
        f"m={m} | "
        f"Rm={R*m:,}"
    )

    print("="*80)


    TR=pack(
        train[:R],
        labels_all[(R,m)]
    )


    net,training_sec=fit_resumable(

        TR,

        VA,

        cfg.seed+7001,

        f"MC_R{R}_m{m}"

    )


    e2,erho=metrics(
        net,
        TE
    )


    progress["mc"][key]={

        "R":
            R,

        "m":
            m,

        "budget":
            R*m,

        "E2_med":
            float(
                np.median(e2)
            ),

        "E2_q1":
            float(
                np.quantile(e2,.25)
            ),

        "E2_q3":
            float(
                np.quantile(e2,.75)
            ),

        "E2_p95":
            float(
                np.quantile(e2,.95)
            ),

        "Erho_med":
            float(
                np.median(erho)
            ),

        "Erho_q1":
            float(
                np.quantile(erho,.25)
            ),

        "Erho_q3":
            float(
                np.quantile(erho,.75)
            ),

        "Erho_p95":
            float(
                np.quantile(erho,.95)
            )

    }


    # Permanent result checkpoint
    atomic_pickle(
        progress,
        PROGRESS_FILE
    )


    print(
        f"{key}: RESULT permanently saved."
    )


    print(
        f"E2 median="
        f"{np.median(e2):.6g} | "
        f"Erho median="
        f"{np.median(erho):.6g}"
    )


    del TR,net


# =====================================================================================
# 24. SINGLE FINAL SCIENTIFIC TABLE
# =====================================================================================

table_rows=[

    {

        "Teacher":
            "Exact",

        "R":
            5000,

        "m":
            np.nan,

        "R_times_m":
            np.nan,

        **EX

    }

]


for R,m in ALLOC:

    key=f"R{R}_m{m}"


    if key in progress["mc"]:

        z=progress["mc"][key]


        table_rows.append(

            {

                "Teacher":
                    "Monte Carlo",

                "R":
                    R,

                "m":
                    m,

                "R_times_m":
                    R*m,

                "E2_med":
                    z["E2_med"],

                "E2_q1":
                    z["E2_q1"],

                "E2_q3":
                    z["E2_q3"],

                "E2_p95":
                    z["E2_p95"],

                "Erho_med":
                    z["Erho_med"],

                "Erho_q1":
                    z["Erho_q1"],

                "Erho_q3":
                    z["Erho_q3"],

                "Erho_p95":
                    z["Erho_p95"]

            }

        )


df=pd.DataFrame(
    table_rows
)


df.to_csv(
    OUT/"coverage_vs_precision.csv",
    index=False
)


(
    OUT/"coverage_vs_precision.tex"
).write_text(

    df.to_latex(
        index=False,
        float_format="%.4g"
    )

)


print("\n"+"="*100)
print("FINAL TABLE")
print("="*100)

print(
    df.to_string(
        index=False
    )
)


# =====================================================================================
# 25. SINGLE FINAL SCIENTIFIC FIGURE
#
# Panel A: median E2 + IQR
# Panel B: median Erho + IQR
#
# Exact reference = horizontal dashed line
# m values annotated
# =====================================================================================

mc=df[
    df["Teacher"]
    ==
    "Monte Carlo"
].copy()


if len(mc):

    fig,ax=plt.subplots(
        1,
        2,
        figsize=(10.5,4.2)
    )


    settings=[

        (
            ax[0],
            "E2_med",
            "E2_q1",
            "E2_q3",
            EX["E2_med"],
            r"$E_2$",
            "(A) Distributional error"
        ),

        (
            ax[1],
            "Erho_med",
            "Erho_q1",
            "Erho_q3",
            EX["Erho_med"],
            r"$E_\rho$",
            "(B) Tail-risk error"
        )

    ]


    for (
        a,
        y,
        q1,
        q3,
        exact_ref,
        ylabel,
        title
    ) in settings:


        x=mc["R"].to_numpy(
            dtype=float
        )


        med=mc[y].to_numpy(
            dtype=float
        )


        lo=mc[q1].to_numpy(
            dtype=float
        )


        hi=mc[q3].to_numpy(
            dtype=float
        )


        a.plot(

            x,
            med,

            "o-",

            color="#0072B2",

            lw=2,

            ms=6,

            label=r"MC, fixed $Rm=50{,}000$"

        )


        a.fill_between(

            x,
            lo,
            hi,

            color="#0072B2",

            alpha=.15,

            linewidth=0

        )


        a.axhline(

            exact_ref,

            color="#D55E00",

            ls="--",

            lw=2,

            label=r"Exact teacher, $R=5000$"

        )


        for xi,yi,mi in zip(
            x,
            med,
            mc["m"]
        ):

            a.annotate(

                rf"$m={int(mi)}$",

                (
                    xi,
                    yi
                ),

                xytext=(
                    0,
                    8
                ),

                textcoords="offset points",

                ha="center",

                fontsize=8

            )


        a.set_xscale(
            "log"
        )


        a.set_xticks(
            x
        )


        a.set_xticklabels(
            x.astype(int)
        )


        a.set_xlabel(
            r"Training configurations $R$"
        )


        a.set_ylabel(
            ylabel
        )


        a.set_title(
            title
        )


        a.grid(
            alpha=.15
        )


        a.legend(
            frameon=False,
            fontsize=9
        )


    plt.tight_layout()


    plt.savefig(
        OUT/"coverage_vs_precision.pdf",
        bbox_inches="tight"
    )


    plt.show()


# =====================================================================================
# 26. FINAL STATUS
# =====================================================================================

print("\n"+"="*90)
print("5.3-B STATUS")
print("="*90)

print(
    "Persistent Google Drive folder:"
)

print(
    ROOT
)

print()


print(
    "Exact teacher completed:",
    progress["exact"] is not None
)


print(
    "Completed MC allocations:",
    len(
        progress["mc"]
    ),
    "/",
    len(
        ALLOC
    )
)


for R,m in ALLOC:

    key=f"R{R}_m{m}"

    print(
        f"{key:15s}:",
        "DONE"
        if key in progress["mc"]
        else "NOT YET"
    )


print()

print(
    "Nominal nested MC trajectories:",
    f"{160_000:,}"
)


print(
    "Overflow early termination:",
    "ON"
)


print(
    "Scientific table:",
    OUT/"coverage_vs_precision.tex"
)


print(
    "Scientific figure:",
    OUT/"coverage_vs_precision.pdf"
)


print(
    "\nAfter a Colab disconnect:"
)

print(
    "1. Open notebook."
)

print(
    "2. Run THIS SAME CELL."
)

print(
    "3. The experiment resumes automatically."
)

print("="*90)

Mounted at /content/drive
PERSISTENT DIRECTORY
/content/drive/MyDrive/StatisticalLearning/Experiment_5_3B_budget50000_v3
Migrated to Drive: exact_train_R5000.pkl
Migrated to Drive: exact_val_400.pkl
Migrated to Drive: exact_test_700.pkl
Migrated to Drive: model_exact_R5000_CPU.pt
CPU cores: 2
Exact workers: 2
Numba threads: 2
PyTorch threads: 2

Generating deterministic designs...

Exact targets...
TRAIN: full Drive cache loaded (5,000)
VAL: full Drive cache loaded (400)
TEST: full Drive cache loaded (700)

EXACT TEACHER | R=5000
exact_R5000 | epoch=  1 | val=1.9307e-01 | best=1.9307e-01 | wait=0 | checkpoint saved
exact_R5000 | epoch=  5 | val=1.2581e-01 | best=1.2391e-01 | wait=1 | checkpoint saved
